In [68]:
from pathlib import Path
import numpy as np
import finitewave as fw


path = Path("/Users/arstanbekokenov/Projects/Finitewave/examples/data/atrial_mesh")

points = np.load(path / "coarse_points.npy") / 2000
elems = np.load(path / "coarse_elems.npy")


tissue = fw.CardiacTissueElements(points, elems, elem_type=fw.ElementType.TRIANGLE)

spatial_discretization = fw.FiniteElementDiscretization()
K, M = spatial_discretization.compute_weights(tissue)

In [69]:
print(points.min(axis=0), points.max(axis=0))

[-44.85182432 -28.79770064  19.02642326] [11.74870276  1.30981202 56.80852586]


In [76]:
stim_sequence = fw.StimSequence()
stim_sequence.add_stim(fw.StimCurrentElectrodes(0, 10, 0.5, points[10:11], size=2))
stim_sequence.add_stim(fw.StimCurrentElectrodes(45, 10, 1.0, points[1000:1001], size=2))   

# create model object and set up parameters:
simulation = fw.CardiacSimulation(backend="jax")
simulation.dt = 0.01
simulation.t_max = 150
# add the tissue and the stim parameters to the model object:
simulation.cardiac_tissue = tissue
simulation.cardiac_model = fw.AlievPanfilov()
simulation.stim_sequence = stim_sequence
# set up the solver:
simulation.solver = fw.BackwardEulerTimeIntegrator(atol=1e-8, maxiter=100)

# run the model:
simulation.run()

# get the resulting potential at the element centers:
u = simulation.cardiac_model.output("u")

Running AlievPanfilov on Triangle Elements: 100%|██████████| 15000/15000 [00:16<00:00, 918.84it/s]


In [77]:
atrial_grid = fw.PyVistaSurfaceGrid(points, elems)
atrial_grid["u"] = u
atrial_grid.plot(cmap="coolwarm")

Widget(value='<iframe src="http://localhost:51679/index.html?ui=P_0x3ae4acf50_26&reconnect=auto" class="pyvist…